# 02 — ROI Detection: Faster R-CNN (Detectron2)

Detect the "Reading Digit" window (ROI) using Faster R-CNN with FPN backbone (Detectron2).
Two experiments: utility-meter dataset (sparse ROI, COCO format) and waterMeterDataset (polygon ROI).

In [1]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        from google.colab import drive
        IN_COLAB = True
    except ImportError:
        pass

if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")

    token = userdata.get("GITHUB_TOKEN", "")
    base = f"https://{token}@github.com" if token else "https://github.com"
    if not Path("/content/WaterMeterCV").exists():
        subprocess.run(
            ["git", "clone", f"{base}/UrranQx/WaterMeterCV.git", "/content/WaterMeterCV"],
            check=True
        )

    BRANCH = "feature/roi-detection"
    subprocess.run(["git", "-C", "/content/WaterMeterCV", "checkout", BRANCH], check=True)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/facebookresearch/detectron2.git",
         "rapidfuzz", "shapely"],
        check=True
    )

    ROOT         = Path("/content/WaterMeterCV")
    DATA_ROOT    = Path("/content/drive/MyDrive/WaterMeterCV/WaterMetricsDATA")
    WEIGHTS_BASE = Path("/content/drive/MyDrive/WaterMeterCV/weights")
    RESULTS_DIR  = Path("/content/drive/MyDrive/WaterMeterCV/results")
    WORKERS = 2
else:
    ROOT         = Path("../..").resolve()
    DATA_ROOT    = ROOT / "WaterMetricsDATA"
    WEIGHTS_BASE = ROOT / "models/weights"
    RESULTS_DIR  = ROOT / "results"
    WORKERS = 0

sys.path.insert(0, str(ROOT))
WEIGHTS_BASE.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import json
import time
import csv
import os
import torch
import cv2
import numpy as np
from datetime import datetime

import detectron2
from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.structures import BoxMode

from models.data.unified_loader import load_water_meter_dataset_split
from models.data.roi_dataset import polygon_to_bbox
from models.metrics.evaluation import compute_iou_bbox

print(f"ROOT: {ROOT}")
print(f"Detectron2: {detectron2.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

C:\wmd2\detectron2\model_zoo\model_zoo.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


ROOT: C:\Users\alike\WaterMeterCV
Detectron2: 0.6
CUDA available: True
GPU: NVIDIA GeForce GTX 1660 Ti


In [2]:
COCO_ROOT = DATA_ROOT / "utility-meter-reading-dataset-for-automatic-reading-yolo.v4i.coco"
WM_PATH = DATA_ROOT / "waterMeterDataset/WaterMeters"

WEIGHTS_DIR = WEIGHTS_BASE / "roi_faster_rcnn"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

MAX_ITER_UM = 3000
MAX_ITER_WM = 5000
BASE_LR = 0.0025
IMS_PER_BATCH = 4
ROI_BATCH_SIZE = 128

RUN_NAME = f"frcnn_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print(f"Run: {RUN_NAME}")

Run: frcnn_20260413_120656


In [3]:
def get_um_roi_dicts(split: str):
    """Load COCO annotations, filter to 'Reading Digit' (category 11)."""
    json_path = COCO_ROOT / split / "_annotations.coco.json"
    with open(json_path) as f:
        coco = json.load(f)

    roi_cat_id = next(c["id"] for c in coco["categories"] if c["name"] == "Reading Digit")
    img_lookup = {img["id"]: img for img in coco["images"]}

    anns_by_img = {}
    for ann in coco["annotations"]:
        if ann["category_id"] == roi_cat_id:
            anns_by_img.setdefault(ann["image_id"], []).append(ann)

    dataset_dicts = []
    for img_id, anns in anns_by_img.items():
        img_info = img_lookup[img_id]
        record = {
            "file_name": str(COCO_ROOT / split / img_info["file_name"]),
            "image_id": img_id,
            "height": img_info["height"],
            "width": img_info["width"],
            "annotations": [],
        }
        for ann in anns:
            record["annotations"].append({
                "bbox": ann["bbox"],
                "bbox_mode": BoxMode.XYWH_ABS,
                "category_id": 0,
            })
        dataset_dicts.append(record)
    return dataset_dicts


def get_wm_roi_dicts(samples):
    """Convert WM samples to Detectron2-format dicts."""
    dataset_dicts = []
    for i, s in enumerate(samples):
        if s.roi_polygon is None:
            continue
        img = cv2.imread(str(s.image_path))
        if img is None:
            continue
        h, w = img.shape[:2]

        cx, cy, bw, bh = polygon_to_bbox(s.roi_polygon)
        x = (cx - bw / 2) * w
        y = (cy - bh / 2) * h

        record = {
            "file_name": str(s.image_path),
            "image_id": i,
            "height": h,
            "width": w,
            "annotations": [{
                "bbox": [x, y, bw * w, bh * h],
                "bbox_mode": BoxMode.XYWH_ABS,
                "category_id": 0,
            }],
        }
        dataset_dicts.append(record)
    return dataset_dicts


# Register UM datasets
for split in ["train", "valid", "test"]:
    name = f"um_roi_{split}"
    if name in DatasetCatalog:
        DatasetCatalog.remove(name)
    DatasetCatalog.register(name, lambda s=split: get_um_roi_dicts(s))
    MetadataCatalog.get(name).set(thing_classes=["ROI"])

# WM split
wm_train, wm_test = load_water_meter_dataset_split(WM_PATH, train_ratio=0.7, seed=42)

for split_name, samples in [("train", wm_train), ("test", wm_test)]:
    name = f"wm_roi_{split_name}"
    if name in DatasetCatalog:
        DatasetCatalog.remove(name)
    DatasetCatalog.register(name, lambda s=samples: get_wm_roi_dicts(s))
    MetadataCatalog.get(name).set(thing_classes=["ROI"])

print(f"UM: {len(get_um_roi_dicts('train'))} train, {len(get_um_roi_dicts('test'))} test")
print(f"WM: {len(wm_train)} train, {len(wm_test)} test")

UM: 45 train, 6 test
WM: 870 train, 374 test


## Experiment 1: utility-meter dataset

In [4]:
cfg_um = get_cfg()
cfg_um.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))
cfg_um.DATASETS.TRAIN = ("um_roi_train",)
cfg_um.DATASETS.TEST  = ("um_roi_valid",)
cfg_um.DATALOADER.NUM_WORKERS = WORKERS
cfg_um.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml")
cfg_um.MODEL.ROI_HEADS.NUM_CLASSES = 1
cfg_um.SOLVER.IMS_PER_BATCH = IMS_PER_BATCH
cfg_um.SOLVER.BASE_LR = BASE_LR
cfg_um.SOLVER.MAX_ITER = MAX_ITER_UM
cfg_um.SOLVER.STEPS = (2000, 2500)
cfg_um.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = ROI_BATCH_SIZE
cfg_um.OUTPUT_DIR = str(WEIGHTS_DIR / f"um_{RUN_NAME}")

os.makedirs(cfg_um.OUTPUT_DIR, exist_ok=True)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

trainer_um = DefaultTrainer(cfg_um)
trainer_um.resume_or_load(resume=False)
trainer_um.train()
print(f"UM training complete. Weights: {cfg_um.OUTPUT_DIR}")

[04/13 12:06:56 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res

C:\wmd2\detectron2\checkpoint\detection_checkpoint.py:73: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  data = pickle.load(f, encoding="latin1")
Skip loading parameter 'roi_heads.box_predictor.cls_score.weight' to the model due to incompatible shapes: (81, 1024) in the checkpoint but (2, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.cls_score.bias' to the model due to incompatible shapes: (81,) in the checkpoint but (2,) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.weight' to the model due to incompatible shapes: (320, 1024) in the checkpoint but (4, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.bias' to the model d

[04/13 12:06:57 d2.engine.train_loop]: Starting training from iteration 0


c:\Users\alike\WaterMeterCV\.venv\Lib\site-packages\torch\functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4383.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W0413 12:06:58.304000 19028 .venv\Lib\site-packages\torch\fx\_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


[04/13 12:07:16 d2.utils.events]:  eta: 0:41:47  iter: 19  total_loss: 2.407  loss_cls: 0.6262  loss_box_reg: 0.145  loss_rpn_cls: 1.207  loss_rpn_loc: 0.3742    time: 0.9264  last_time: 0.7533  data_time: 0.1086  last_data_time: 0.0274   lr: 4.9952e-05  max_mem: 4021M
[04/13 12:07:33 d2.utils.events]:  eta: 0:39:17  iter: 39  total_loss: 1.164  loss_cls: 0.4253  loss_box_reg: 0.3783  loss_rpn_cls: 0.08829  loss_rpn_loc: 0.2283    time: 0.9003  last_time: 0.7497  data_time: 0.0854  last_data_time: 0.0567   lr: 9.9902e-05  max_mem: 4021M
[04/13 12:07:52 d2.utils.events]:  eta: 0:39:22  iter: 59  total_loss: 1.009  loss_cls: 0.3392  loss_box_reg: 0.3857  loss_rpn_cls: 0.09983  loss_rpn_loc: 0.1862    time: 0.9152  last_time: 1.6291  data_time: 0.1046  last_data_time: 0.0572   lr: 0.00014985  max_mem: 4021M
[04/13 12:08:10 d2.utils.events]:  eta: 0:38:45  iter: 79  total_loss: 1.084  loss_cls: 0.3434  loss_box_reg: 0.4648  loss_rpn_cls: 0.07473  loss_rpn_loc: 0.1807    time: 0.9005  last_

## Evaluation (utility-meter)

In [5]:
cfg_um.MODEL.WEIGHTS = os.path.join(cfg_um.OUTPUT_DIR, "model_final.pth")
cfg_um.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
um_predictor = DefaultPredictor(cfg_um)

um_test_dicts = get_um_roi_dicts("test")
um_ious = []
um_inference_times = []
for d in um_test_dicts:
    img = cv2.imread(d["file_name"])
    h, w = img.shape[:2]

    t0 = time.perf_counter()
    outputs = um_predictor(img)
    um_inference_times.append((time.perf_counter() - t0) * 1000)

    pred_boxes = outputs["instances"].pred_boxes.tensor.cpu().numpy()
    if len(pred_boxes) > 0:
        px1, py1, px2, py2 = pred_boxes[0]
        pred_cxcywh = ((px1+px2)/(2*w), (py1+py2)/(2*h), (px2-px1)/w, (py2-py1)/h)
        gt = d["annotations"][0]["bbox"]  # [x, y, w, h] absolute
        gt_cxcywh = ((gt[0]+gt[2]/2)/w, (gt[1]+gt[3]/2)/h, gt[2]/w, gt[3]/h)
        um_ious.append(compute_iou_bbox(pred_cxcywh, gt_cxcywh))
    else:
        um_ious.append(0.0)

um_mean_iou = np.mean(um_ious) if um_ious else 0.0
um_det_rate = sum(1 for v in um_ious if v > 0) / len(um_ious) if um_ious else 0.0
um_avg_ms   = np.mean(um_inference_times) if um_inference_times else 0.0

print(f"UM — Mean IoU:        {um_mean_iou:.4f}")
print(f"UM — Detection rate:  {um_det_rate:.4f} ({sum(1 for v in um_ious if v > 0)}/{len(um_ious)})")
print(f"UM — Avg inference:   {um_avg_ms:.1f} ms/image")

[04/13 13:08:11 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from C:\Users\alike\WaterMeterCV\models\weights\roi_faster_rcnn\um_frcnn_20260413_120656\model_final.pth ...
UM — Mean IoU:        0.1375
UM — Detection rate:  0.1667 (1/6)
UM — Avg inference:   145.1 ms/image


## Experiment 2: waterMeterDataset

In [6]:
cfg_wm = get_cfg()
cfg_wm.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))
cfg_wm.DATASETS.TRAIN = ("wm_roi_train",)
cfg_wm.DATASETS.TEST  = ("wm_roi_test",)
cfg_wm.DATALOADER.NUM_WORKERS = WORKERS
cfg_wm.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml")
cfg_wm.MODEL.ROI_HEADS.NUM_CLASSES = 1
cfg_wm.SOLVER.IMS_PER_BATCH = IMS_PER_BATCH
cfg_wm.SOLVER.BASE_LR = BASE_LR
cfg_wm.SOLVER.MAX_ITER = MAX_ITER_WM
cfg_wm.SOLVER.STEPS = (3000, 4000)
cfg_wm.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = ROI_BATCH_SIZE
cfg_wm.OUTPUT_DIR = str(WEIGHTS_DIR / f"wm_{RUN_NAME}")

os.makedirs(cfg_wm.OUTPUT_DIR, exist_ok=True)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

trainer_wm = DefaultTrainer(cfg_wm)
trainer_wm.resume_or_load(resume=False)
trainer_wm.train()
print(f"WM training complete. Weights: {cfg_wm.OUTPUT_DIR}")

[04/13 13:08:13 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res

C:\wmd2\detectron2\checkpoint\detection_checkpoint.py:73: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  data = pickle.load(f, encoding="latin1")
Skip loading parameter 'roi_heads.box_predictor.cls_score.weight' to the model due to incompatible shapes: (81, 1024) in the checkpoint but (2, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.cls_score.bias' to the model due to incompatible shapes: (81,) in the checkpoint but (2,) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.weight' to the model due to incompatible shapes: (320, 1024) in the checkpoint but (4, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.bias' to the model d

[04/13 13:08:19 d2.engine.train_loop]: Starting training from iteration 0
[04/13 13:09:05 d2.utils.events]:  eta: 3:10:26  iter: 19  total_loss: 0.8426  loss_cls: 0.6109  loss_box_reg: 0.1306  loss_rpn_cls: 0.06689  loss_rpn_loc: 0.01143    time: 2.4158  last_time: 3.3960  data_time: 0.1056  last_data_time: 0.1235   lr: 4.9952e-05  max_mem: 5508M
[04/13 13:09:47 d2.utils.events]:  eta: 3:10:47  iter: 39  total_loss: 0.5447  loss_cls: 0.2615  loss_box_reg: 0.1928  loss_rpn_cls: 0.05874  loss_rpn_loc: 0.01348    time: 2.2545  last_time: 2.3319  data_time: 0.1077  last_data_time: 0.1082   lr: 9.9902e-05  max_mem: 5508M
[04/13 13:10:29 d2.utils.events]:  eta: 3:08:20  iter: 59  total_loss: 0.5176  loss_cls: 0.2191  loss_box_reg: 0.2715  loss_rpn_cls: 0.01365  loss_rpn_loc: 0.01212    time: 2.1993  last_time: 2.2295  data_time: 0.0985  last_data_time: 0.0507   lr: 0.00014985  max_mem: 5508M
[04/13 13:11:11 d2.utils.events]:  eta: 3:08:05  iter: 79  total_loss: 0.5636  loss_cls: 0.2092  loss

In [7]:
cfg_wm.MODEL.WEIGHTS = os.path.join(cfg_wm.OUTPUT_DIR, "model_final.pth")
cfg_wm.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
wm_predictor = DefaultPredictor(cfg_wm)

wm_test_dicts = get_wm_roi_dicts(wm_test)
wm_ious = []
wm_inference_times = []
for d in wm_test_dicts:
    img = cv2.imread(d["file_name"])
    h, w = img.shape[:2]

    t0 = time.perf_counter()
    outputs = wm_predictor(img)
    wm_inference_times.append((time.perf_counter() - t0) * 1000)

    pred_boxes = outputs["instances"].pred_boxes.tensor.cpu().numpy()
    if len(pred_boxes) > 0:
        px1, py1, px2, py2 = pred_boxes[0]
        pred_cxcywh = ((px1+px2)/(2*w), (py1+py2)/(2*h), (px2-px1)/w, (py2-py1)/h)
        gt = d["annotations"][0]["bbox"]
        gt_cxcywh = ((gt[0]+gt[2]/2)/w, (gt[1]+gt[3]/2)/h, gt[2]/w, gt[3]/h)
        wm_ious.append(compute_iou_bbox(pred_cxcywh, gt_cxcywh))
    else:
        wm_ious.append(0.0)

wm_mean_iou = np.mean(wm_ious) if wm_ious else 0.0
wm_det_rate = sum(1 for v in wm_ious if v > 0) / len(wm_ious) if wm_ious else 0.0
wm_avg_ms   = np.mean(wm_inference_times) if wm_inference_times else 0.0

print(f"WM — Mean IoU:        {wm_mean_iou:.4f}")
print(f"WM — Detection rate:  {wm_det_rate:.4f} ({sum(1 for v in wm_ious if v > 0)}/{len(wm_ious)})")
print(f"WM — Avg inference:   {wm_avg_ms:.1f} ms/image")

[04/13 16:29:11 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from C:\Users\alike\WaterMeterCV\models\weights\roi_faster_rcnn\wm_frcnn_20260413_120656\model_final.pth ...
WM — Mean IoU:        0.9397
WM — Detection rate:  1.0000 (374/374)
WM — Avg inference:   405.4 ms/image


## Predictions

In [8]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# Top row: UM test
for i, ax in enumerate(axes[0]):
    if i >= len(um_test_dicts):
        ax.axis("off")
        continue
    d = um_test_dicts[i]
    img = cv2.imread(d["file_name"])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h_img, w_img = img.shape[:2]

    gt = d["annotations"][0]["bbox"]
    gx, gy, gw, gh = int(gt[0]), int(gt[1]), int(gt[2]), int(gt[3])
    cv2.rectangle(img, (gx, gy), (gx+gw, gy+gh), (0, 255, 0), 2)

    outputs = um_predictor(cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    pred_boxes = outputs["instances"].pred_boxes.tensor.cpu().numpy()
    if len(pred_boxes) > 0:
        px1, py1, px2, py2 = [int(v) for v in pred_boxes[0]]
        cv2.rectangle(img, (px1, py1), (px2, py2), (255, 0, 0), 2)

    ax.imshow(img)
    iou_val = um_ious[i] if i < len(um_ious) else 0
    ax.set_title(f"UM IoU={iou_val:.2f}", fontsize=10)
    ax.axis("off")

# Bottom row: WM test
for i, ax in enumerate(axes[1]):
    if i >= len(wm_test_dicts):
        ax.axis("off")
        continue
    d = wm_test_dicts[i]
    img = cv2.imread(d["file_name"])
    if img is None:
        ax.axis("off")
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    gt = d["annotations"][0]["bbox"]
    gx, gy, gw, gh = int(gt[0]), int(gt[1]), int(gt[2]), int(gt[3])
    cv2.rectangle(img, (gx, gy), (gx+gw, gy+gh), (0, 255, 0), 2)

    outputs = wm_predictor(cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    pred_boxes = outputs["instances"].pred_boxes.tensor.cpu().numpy()
    if len(pred_boxes) > 0:
        px1, py1, px2, py2 = [int(v) for v in pred_boxes[0]]
        cv2.rectangle(img, (px1, py1), (px2, py2), (255, 0, 0), 2)

    ax.imshow(img)
    iou_val = wm_ious[i] if i < len(wm_ious) else 0
    ax.set_title(f"WM IoU={iou_val:.2f}", fontsize=10)
    ax.axis("off")

plt.suptitle("Faster R-CNN ROI — Green=GT, Red=Pred", fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "roi_faster_rcnn_predictions.png", dpi=150)
plt.close()
print("Saved to results/roi_faster_rcnn_predictions.png")

Saved to results/roi_faster_rcnn_predictions.png


## Comparison

In [9]:
print(f"{'='*60}")
print(f"{'Metric':<25} {'utility-meter':>15} {'waterMeter':>15}")
print(f"{'='*60}")
print(f"{'Mean IoU':<25} {um_mean_iou:>15.4f} {wm_mean_iou:>15.4f}")
print(f"{'Detection rate':<25} {um_det_rate:>15.4f} {wm_det_rate:>15.4f}")
print(f"{'Avg inference (ms)':<25} {um_avg_ms:>15.1f} {wm_avg_ms:>15.1f}")
print(f"{'N test':<25} {len(um_test_dicts):>15d} {len(wm_test_dicts):>15d}")
print(f"{'='*60}")

Metric                      utility-meter      waterMeter
Mean IoU                           0.1375          0.9397
Detection rate                     0.1667          1.0000
Avg inference (ms)                  145.1           405.4
N test                                  6             374


## Save Results

In [12]:
metrics = {
    "method": "faster_rcnn",
    "utility_meter": {
        "mean_iou": round(float(um_mean_iou), 4),
        "detection_rate": round(float(um_det_rate), 4),
        "avg_inference_ms": round(float(um_avg_ms), 1),
        "n_train": int(len(get_um_roi_dicts("train"))),
        "n_test": int(len(um_test_dicts)),
    },
    "water_meter": {
        "mean_iou": round(float(wm_mean_iou), 4),
        "detection_rate": round(float(wm_det_rate), 4),
        "avg_inference_ms": round(float(wm_avg_ms), 1),
        "n_train": int(len(wm_train)),
        "n_test": int(len(wm_test)),
    },
    "config": {
        "max_iter_um": int(MAX_ITER_UM),
        "max_iter_wm": int(MAX_ITER_WM),
        "base_lr": float(BASE_LR),
        "ims_per_batch": int(IMS_PER_BATCH),
    },
    "run_date": datetime.now().isoformat(),
}

with open(RESULTS_DIR / "roi_faster_rcnn_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

csv_path = RESULTS_DIR / "roi_comparison.csv"
csv_exists = csv_path.exists()
with open(csv_path, "a", newline="") as f:
    writer = csv.writer(f)
    if not csv_exists:
        writer.writerow([
            "method", "um_mean_iou", "um_detection_rate", "um_inference_ms",
            "wm_mean_iou", "wm_detection_rate", "wm_inference_ms", "run_date",
        ]) 
    writer.writerow([
        "faster_rcnn",
        round(um_mean_iou, 4), round(um_det_rate, 4), round(um_avg_ms, 1),
        round(wm_mean_iou, 4), round(wm_det_rate, 4), round(wm_avg_ms, 1),
        datetime.now().strftime("%Y-%m-%d %H:%M"),
    ])

print(f"Metrics -> {RESULTS_DIR / 'roi_faster_rcnn_metrics.json'}")
print(f"CSV    -> {csv_path}")

Metrics -> C:\Users\alike\WaterMeterCV\results\roi_faster_rcnn_metrics.json
CSV    -> C:\Users\alike\WaterMeterCV\results\roi_comparison.csv


## Conclusions

*(Fill after running)*

- **Faster R-CNN (utility-meter):** Mean IoU=..., Detection rate=...
- **Faster R-CNN (waterMeter):** Mean IoU=..., Detection rate=...
- **Inference:** ... ms/image
- **Next step:** compare with YOLO and U-Net